# Pipeline de expresiones faciales equinas

Este notebook **ejecuta los demás en orden**. Es el único que hace falta abrir para correr el
pipeline completo.

Resuelve el problema que tenía el pipeline partido en notebooks sueltos: cada notebook de Colab
arrancaba con un `/content` vacío, así que el `02` no encontraba los clips y los CSV que había
generado el `01`. Ahora todos los artefactos viven en una carpeta de Google Drive y el paso de
datos entre etapas ocurre por ahí.

Qué hace, concretamente:

- **saltea** las etapas cuyos resultados ya están en Drive, para poder avanzar de a tramos a lo
  largo de varias sesiones;
- **corta la cadena** si una etapa falla, en vez de seguir con las que dependen de ella;
- **guarda una copia ejecutada** de cada notebook, con sus salidas y los parámetros usados, como
  evidencia reproducible de qué se corrió y con qué resultado.

Colab corta la sesión por límite de tiempo o por inactividad. Si eso pasa a mitad de camino,
alcanza con volver a ejecutar este notebook: retoma donde había quedado.

## Paso 0 — Arranque: montar Drive, traer el código e instalar papermill

In [ ]:
# ============================================================
# ARRANQUE DEL ORQUESTADOR
# ============================================================
# Autocontenida a propósito: corre antes de que el paquete `equinos` esté disponible.
#
# A diferencia de los notebooks de etapa, éste necesita además los .ipynb de las etapas en
# disco, porque papermill los ejecuta como archivos.
#
# El repo es privado, así que un `git clone` sin credenciales no alcanza. El token va en los
# secrets de Colab (panel izquierdo, icono de llave) y NUNCA dentro del notebook: lo que se
# escribe en una celda queda en el archivo y se comparte con él.

import shutil
import subprocess
import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"
REPO = "github.com/sergioyanez/Tesina-UGR.git"

# True vuelve a traer el código del repo aunque ya esté en Drive. Ponerlo en True después de
# cambiar un notebook o el paquete.
SINCRONIZAR_CODIGO = False

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    CODIGO = Path("/content/drive/MyDrive") / NOMBRE_PROYECTO / "codigo"
    CODIGO.mkdir(parents=True, exist_ok=True)

    if SINCRONIZAR_CODIGO or not (CODIGO / "equinos").is_dir():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None

        if not token:
            raise SystemExit(
                f"Falta el código en {CODIGO}.\n"
                "  Opción A: cargá un secret GITHUB_TOKEN en Colab (panel izquierdo, icono de llave).\n"
                f"  Opción B: copiá a mano `equinos/` y los notebooks NN_*.ipynb dentro de {CODIGO}."
            )

        clon = Path("/content/_repo_tesina")
        shutil.rmtree(clon, ignore_errors=True)
        resultado = subprocess.run(
            ["git", "clone", "--depth", "1", f"https://{token}@{REPO}", str(clon)],
            capture_output=True, text=True,
        )
        if resultado.returncode != 0:
            # No se imprime el comando ni stderr: contienen el token.
            raise SystemExit("Falló el clonado. Verificá que GITHUB_TOKEN tenga permiso de lectura del repo.")

        shutil.copytree(clon / "equinos", CODIGO / "equinos", dirs_exist_ok=True)
        for notebook in sorted(clon.glob("[0-9][0-9]_*.ipynb")):
            shutil.copy(notebook, CODIGO / notebook.name)
        print(f"Código sincronizado en {CODIGO}")

    sys.path.insert(0, str(CODIGO))
else:
    CODIGO = Path.cwd()
    sys.path.insert(0, str(CODIGO))

# ============================================================
# DEPENDENCIAS DE TODO EL PIPELINE
# ============================================================
# Se instalan acá, de una sola vez, ANTES de lanzar ninguna etapa.
#
# Los notebooks de etapa corren en kernels separados pero comparten el site-packages de la
# sesión. Si cada uno instala lo suyo sobre la marcha, el segundo le cambia las versiones al
# primero por debajo. En particular DeepLabCut fija numpy<2, y Colab trae scipy y pandas
# compilados contra numpy 2: instalarlo tarde rompe el entorno entero.
#
# Instalando el conjunto completo antes de arrancar, cada kernel de etapa se encuentra un disco
# ya consistente. Este kernel, el del orquestador, no importa numpy ni pandas justamente para
# no quedar atado a esas versiones.

!pip install -q papermill deeplabcut "scipy<1.14" ultralytics opencv-python pandas numpy tqdm matplotlib

from equinos import cache, checkpoint, etapas, rutas

RAIZ = rutas.inicializar()

## Paso 1 — Parámetros compartidos

In [ ]:
# ============================================================
# PARÁMETROS DEL PIPELINE
# ============================================================
# Un solo lugar para los valores que atraviesan varias etapas. Papermill los inyecta en cada
# notebook, así que barrer un umbral no requiere abrir ni editar ninguna celda de las etapas.
#
# Sólo hay que listar lo que se quiera cambiar: lo que no aparezca acá toma el valor por defecto
# declarado en la celda `parameters` del notebook correspondiente.

PARAMETROS = {
    "preparar_dataset": {
        # "DURACION_CLIP": 6.0,
        # "NUM_FRAMES": 16,
    },
    "roi_normalizacion": {
        # "CONF_MINIMA_DETECCION": 0.50,
        # "MARGEN_ROI": 0.45,
        # "COBERTURA_MINIMA_ACEPTABLE": 0.70,
    },
}

# Etapas a re-ejecutar aunque ya estén completas. Ojo: descarta sus artefactos.
ETAPAS_A_FORZAR = []

## Paso 2 — Diagnóstico: en qué estado está el pipeline

No ejecuta ni modifica nada. Conviene mirarlo al abrir una sesión nueva, para saber qué falta.

In [ ]:
reporte = etapas.diagnostico()

## Paso 3 — Ejecutar

Las etapas completas se saltean. Si una falla, el pipeline se detiene ahí: seguir con las que
dependen de ella sólo produciría un segundo error más confuso que el primero.

In [ ]:
import shutil
import time
from datetime import datetime

import papermill as pm

# ============================================================
# EJECUCIÓN ENCADENADA
# ============================================================
# Cada etapa corre en su propio kernel. Las etapas NO se comunican por memoria compartida: el
# único canal entre ellas son los artefactos persistidos en la raíz de proyecto. Es lo que
# permite que una sesión ejecute el 01, se corte, y otra sesión al día siguiente ejecute el 02.

resumen = []
orden = etapas.orden_de_ejecucion()

print(f"Notebooks en : {CODIGO}")
print(f"Artefactos en: {rutas.raiz()}")
print(f"Etapas       : {len(orden)} -> {', '.join(orden)}\n")

for nombre in orden:
    etapa = etapas.PIPELINE[nombre]
    forzar = nombre in ETAPAS_A_FORZAR
    situacion = etapas.estado(nombre)["estado"]

    if situacion == etapas.COMPLETA and not forzar:
        print(f"[=] {nombre:<20} ya completa, se saltea")
        resumen.append({"etapa": nombre, "resultado": "salteada", "segundos": 0})
        continue

    if forzar:
        print(f"[!] {nombre:<20} re-ejecución forzada: se descartan sus artefactos")
        etapas.limpiar(nombre)

    entrada = CODIGO / etapa.notebook
    if not entrada.exists():
        raise SystemExit(f"No se encontró {entrada}. Poné SINCRONIZAR_CODIGO = True y reejecutá el Paso 0.")

    marca = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Papermill escribe la salida celda por celda mientras ejecuta. Se escribe al disco local y
    # se copia a Drive al final: escribir en Drive durante una hora de ejecución es lento y
    # más frágil que hacerlo una sola vez.
    salida_local = rutas.dir_trabajo("ejecuciones") / f"{nombre}_{marca}.ipynb"

    print(f"[>] {nombre:<20} ejecutando {etapa.notebook} ({situacion})")
    inicio = time.time()
    fallo = None

    try:
        pm.execute_notebook(
            str(entrada),
            str(salida_local),
            parameters=PARAMETROS.get(nombre, {}),
            kernel_name="python3",
            progress_bar=False,
            log_output=True,
        )
    except Exception as error:
        fallo = error

    transcurrido = time.time() - inicio
    sufijo = "_FALLIDA" if fallo else ""
    destino = rutas.carpeta("ejecuciones") / f"{nombre}_{marca}{sufijo}.ipynb"
    if salida_local.exists():
        shutil.copy(salida_local, destino)

    if fallo:
        print(f"\n[x] {nombre:<20} FALLÓ tras {transcurrido/60:.1f} min")
        print(f"    {type(fallo).__name__}: {fallo}")
        print(f"    copia con el error: {destino}")
        resumen.append({"etapa": nombre, "resultado": "fallida", "segundos": transcurrido})
        dependientes = [n for n, e in etapas.PIPELINE.items() if nombre in e.depende_de]
        if dependientes:
            print(f"\nSe detiene el pipeline: {', '.join(dependientes)} dependen de esta etapa.")
        break

    print(f"[v] {nombre:<20} OK en {transcurrido/60:.1f} min -> {destino}")
    resumen.append({"etapa": nombre, "resultado": "ejecutada", "segundos": transcurrido})

## Paso 4 — Resumen de la corrida

Si el kernel se reinició, el resumen en memoria se pierde pero el estado real no: está en los
artefactos de Drive, y el diagnóstico del final lo lee de ahí.


In [ ]:
# ============================================================
# RESUMEN DE LA CORRIDA
# ============================================================
# Deliberadamente sin pandas ni numpy.
#
# Los notebooks de etapa corren en kernels separados pero comparten el site-packages de la
# sesión, y algunos instalan sus propias dependencias: DeepLabCut fija su versión de numpy.
# Eso puede dejar el stack científico de ESTE kernel inconsistente con lo que ya tenía cargado,
# y `import pandas` falla con "numpy.dtype size changed".
#
# El orquestador no debe depender de lo que sus hijos mutan. Con la biblioteca estándar alcanza
# de sobra para imprimir una tabla.

if "resumen" in globals() and resumen:
    ancho = max(len(f["etapa"]) for f in resumen)

    print(f"{'ETAPA'.ljust(ancho)}  {'RESULTADO':<10}  MINUTOS")
    print("-" * (ancho + 22))
    for fila in resumen:
        print(f"{fila['etapa'].ljust(ancho)}  {fila['resultado']:<10}  {fila['segundos'] / 60:7.1f}")

    print(f"\nTiempo total: {sum(f['segundos'] for f in resumen) / 60:.1f} min")
else:
    print("Sin resumen en memoria: el kernel se reinició o el Paso 3 no llegó a correr.")
    print("El estado real de cada etapa está abajo, leído de los artefactos en Drive.")

print("\nEstado final del pipeline:\n")
etapas.diagnostico()
